# Egyptora · Monument recognizer

Trains the camera model on 5 monuments + **other**, following the *Monument photos plan* doc.

1. Check and split the photos
2. Baseline with no training (DINOv2 + nearest photos)
3. Fine-tune ConvNeXt-Tiny
4. Calibrate confidence and pick the threshold on **val**
5. Final numbers on **test** (only once, at the end)
6. Export `recognizer.onnx` + `labels.json`

**On Colab:** Runtime → Change runtime type → T4 GPU, then Run all. The photos are read from Google Drive (`MyDrive/Egyptora/monuments`). **On Kaggle:** add the photos as a dataset, turn the GPU on, and set `DATA_ROOT` below.

In [ ]:
%pip install -q timm imagehash onnx onnxruntime

## Setup

On Colab this gets our code from GitHub and connects Google Drive.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
SMOKE = os.environ.get("EGYPTORA_SMOKE") == "1"  # tiny run used to test this notebook, not for real training

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    if not Path("Egyptora").exists():
        subprocess.run(["git", "clone", "-q", "https://github.com/HamdyElbauomi/Egyptora.git"], check=True)

CODE_DIR = Path(os.environ.get("EGYPTORA_CODE", "Egyptora/ml/recognizer" if IN_COLAB else "."))
sys.path.insert(0, str(CODE_DIR))

import matplotlib.pyplot as plt  # noqa: E402
import monuments as m  # noqa: E402
import numpy as np  # noqa: E402
import timm  # noqa: E402

print("device:", m.device())

## Settings

In [ ]:
DATA_ROOT = Path(os.environ.get("EGYPTORA_DATA", "/content/drive/MyDrive/Egyptora/monuments"))
OUT_DIR = DATA_ROOT / "runs" / "convnext_tiny"
OUT_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 224
BATCH_SIZE = 32
MODEL = "convnext_tiny.fb_in22k_ft_in1k"  # pretrained on ImageNet-22k, then 1k
BASELINE_MODEL = "vit_small_patch14_dinov2.lvd142m"
PRETRAINED = True
HEAD_EPOCHS, FULL_EPOCHS = 3, 12

if SMOKE:  # small, untrained models so the whole notebook runs on a CPU in minutes
    IMG_SIZE, BATCH_SIZE = 112, 8
    MODEL, BASELINE_MODEL, PRETRAINED = "convnext_atto", "vit_small_patch14_dinov2", False
    HEAD_EPOCHS, FULL_EPOCHS = 1, 1

## 1 · Check and split the photos

Drops broken, tiny and duplicate photos, and training photos that look like a test photo. Read the warnings before going on.

In [ ]:
result = subprocess.run(
    [sys.executable, str(CODE_DIR / "prepare_data.py"), "--root", str(DATA_ROOT)], capture_output=True, text=True
)
print(result.stdout, result.stderr)
data = m.loaders(DATA_ROOT, IMG_SIZE, BATCH_SIZE)
print({name: len(loader.dataset) for name, loader in data.items()})

## 2 · Baseline: no training

A ready model (DINOv2) turns every photo into a feature vector. Each test photo takes the label of its most similar training photos. Fine-tuning has to beat this.

In [ ]:
backbone = timm.create_model(BASELINE_MODEL, pretrained=PRETRAINED, num_classes=0, img_size=IMG_SIZE)
plain = {
    name: m.DataLoader(m.Photos(loader.dataset.items, m.make_transform(IMG_SIZE, train=False)), batch_size=BATCH_SIZE)
    for name, loader in data.items()
}
train_f, train_y = m.embed(backbone, plain["train"], m.device())
test_f, test_y = m.embed(backbone, plain["test"], m.device())
baseline_probs = m.knn_probs(train_f, train_y, test_f)
baseline = m.report(baseline_probs, test_y, threshold=0.5)
print("baseline accuracy:", baseline["accuracy"], " macro F1:", baseline["macro_f1"])
del backbone

## 3 · Fine-tune

Stage 1 trains only the new last layer. Stage 2 trains the whole model slowly. The best epoch on **val** is kept; training stops early when val stops improving.

In [ ]:
model = m.build_classifier(MODEL, pretrained=PRETRAINED)
history = m.fine_tune(model, data, OUT_DIR / "best.pt", head_epochs=HEAD_EPOCHS, full_epochs=FULL_EPOCHS)
print("best epoch:", history.best_epoch, " val macro F1:", round(history.best_val_f1, 4))

In [ ]:
epochs = [h["epoch"] for h in history.epochs]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(epochs, [h["train_loss"] for h in history.epochs], label="train")
axes[0].plot(epochs, [h["val_loss"] for h in history.epochs], label="val")
axes[0].set_title("Loss")
axes[1].plot(epochs, [h["val_acc"] for h in history.epochs], label="val accuracy")
axes[1].plot(epochs, [h["val_f1"] for h in history.epochs], label="val macro F1")
axes[1].set_title("Validation")
for ax in axes:
    ax.set_xlabel("epoch")
    ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "training_curves.png", dpi=150)
plt.show()

## 4 · Confidence and threshold (on val)

Temperature scaling makes the confidence honest (0.8 should be right about 80% of the time). Then pick the threshold: under it, or when the answer is *other*, the app says **not sure**. Prefer a threshold with few wrong answers, even if a few more photos get *not sure*.

In [ ]:
val_logits, val_y = m.logits_of(model, data["val"])
T = m.fit_temperature(val_logits, val_y)
val_probs = m.softmax(val_logits, T)
print("temperature:", round(T, 3))
for row in m.threshold_table(val_probs, val_y):
    print(row)

In [ ]:
THRESHOLD = 0.6  # change after reading the table above

## 5 · Final numbers on test

Run this once. Don't go back and tune after looking at test.

In [ ]:
test_logits, test_y = m.logits_of(model, data["test"])
test_probs = m.softmax(test_logits, T)
final = m.report(test_probs, test_y, THRESHOLD)
print(f"{'':<16}{'baseline':>10}{'fine-tuned':>12}")
for key in ("accuracy", "macro_f1"):
    print(f"{key:<16}{baseline[key]:>10}{final[key]:>12}")
print()
for key in ("monuments_recognized_correctly", "monuments_wrong_answer", "other_photos_rejected"):
    print(f"{key:<34}{final[key]}")

In [ ]:
cm = np.array(final["confusion"])
fig, ax = plt.subplots(figsize=(6.5, 5.5))
ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(m.LABELS)), m.LABELS, rotation=40, ha="right")
ax.set_yticks(range(len(m.LABELS)), m.LABELS)
ax.set_xlabel("predicted")
ax.set_ylabel("true")
for i in range(len(m.LABELS)):
    for j in range(len(m.LABELS)):
        ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set_title("Test confusion matrix")
plt.tight_layout()
plt.savefig(OUT_DIR / "confusion_matrix.png", dpi=150)
plt.show()

Mistakes are the most useful thing to look at: they show which photos to collect more of.

In [ ]:
from PIL import Image  # noqa: E402

pred = test_probs.argmax(1)
wrong = np.flatnonzero(pred != test_y)[:12]
if len(wrong):
    cols = 4
    rows = (len(wrong) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(3.2 * cols, 3.4 * rows), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, i in zip(axes.flat, wrong, strict=False):
        path = data["test"].dataset.items[i][0]
        ax.imshow(Image.open(path).convert("RGB"))
        ax.set_title(f"true: {m.LABELS[test_y[i]]}\npred: {m.LABELS[pred[i]]} {test_probs[i].max():.2f}", fontsize=9)
    plt.tight_layout()
    plt.show()
else:
    print("no mistakes on test")

## 6 · Export

`recognizer.onnx` runs on the server's CPU without PyTorch. `labels.json` holds the labels, image settings, temperature, threshold and the test numbers. Share both with the team, then we plug them into `app/ai/recognizer.py` together.

In [ ]:
onnx_path, labels_path = m.export(model, OUT_DIR, IMG_SIZE, T, THRESHOLD, final)
print("max difference ONNX vs PyTorch:", m.check_onnx(onnx_path, model, IMG_SIZE))
print("saved:", onnx_path, labels_path)